# 🔧 Code Repair Fine-Tuning
**YZM358 — Code Review Chatbot | Üye 2**

Bu notebook CodeT5+ modelini **kod düzeltme (code repair)** görevi için eğitir.
- Veri: `microsoft/codexglue_code_refinement` (small split)
- Model: `repair_model/` klasörüne kaydedilir, mevcut `final_model/` korunur
- `fp16=False` (T4 GPU uyumu için)

**Hücreleri sırayla çalıştırın!**

In [1]:
# ============================================================
# ADIM 1: Kütüphaneler
# ============================================================
!pip install -q transformers datasets sentencepiece
print('✅ Kütüphaneler güncellendi!')

print('✅ Kütüphaneler hazır!')

✅ Kütüphaneler güncellendi!
✅ Kütüphaneler hazır!


In [2]:
# ============================================================
# ADIM 2: Drive Bağla
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_KOKU   = '/content/drive/MyDrive/CodeReviewBot'
TOKENIZER_YOLU = f'{DRIVE_KOKU}/model_ciktisi/final_model'  # mevcut tokenizer
REPAIR_YOLU  = f'{DRIVE_KOKU}/model_ciktisi/repair_model'   # yeni model buraya
os.makedirs(REPAIR_YOLU, exist_ok=True)

print(f'✅ Drive bağlandı!')
print(f'   Tokenizer kaynağı: {TOKENIZER_YOLU}')
print(f'   Repair model hedefi: {REPAIR_YOLU}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive bağlandı!
   Tokenizer kaynağı: /content/drive/MyDrive/CodeReviewBot/model_ciktisi/final_model
   Repair model hedefi: /content/drive/MyDrive/CodeReviewBot/model_ciktisi/repair_model


In [3]:
from datasets import load_dataset

print('CodeXGLUE Code Refinement indiriliyor...')

try:
    dataset = load_dataset("google-research-datasets/codexglue_code_refinement", "small")
except:
    try:
        dataset = load_dataset("code_x_glue_cc_code_refinement", "small")
    except:
        dataset = load_dataset("microsoft/CodeXGLUE", "code-refinement-small")

print(f'\n✅ Veri yüklendi!')
print(f'  Train : {len(dataset["train"])} örnek')
print(f'  Valid : {len(dataset["validation"])} örnek')
print(f'  Test  : {len(dataset["test"])} örnek')
print(f'  Sütunlar: {dataset["train"].column_names}')

print('\n📋 İlk 2 örnek:')
for i in range(2):
    ornek = dataset['train'][i]
    print(f'--- Örnek {i+1} ---')
    print(f'  buggy: {str(ornek.get("buggy", ornek.get("input", "")))[:100]}...')
    print(f'  fixed: {str(ornek.get("fixed", ornek.get("output", "")))[:100]}...')

CodeXGLUE Code Refinement indiriliyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



✅ Veri yüklendi!
  Train : 46680 örnek
  Valid : 5835 örnek
  Test  : 5835 örnek
  Sütunlar: ['id', 'buggy', 'fixed']

📋 İlk 2 örnek:
--- Örnek 1 ---
  buggy: public java.lang.String METHOD_1 ( ) { return new TYPE_1 ( STRING_1 ) . format ( VAR_1 [ ( ( VAR_1 ....
  fixed: public java.lang.String METHOD_1 ( ) { return new TYPE_1 ( STRING_1 ) . format ( VAR_1 [ ( ( type ) ...
--- Örnek 2 ---
  buggy: public boolean METHOD_1 ( java.lang.String name ) { TYPE_1 VAR_1 = TYPE_1 . METHOD_2 ( VAR_2 ) ; ret...
  fixed: public boolean METHOD_1 ( java.lang.String name ) { return ( ! ( METHOD_3 ( name ) ) ) && ( TYPE_1 ....


In [5]:
import torch # Önce import etmeliyiz
from transformers import RobertaTokenizer, T5ForConditionalGeneration

# Eski modeli temizle
if 'model' in locals(): del model
torch.cuda.empty_cache()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device.upper()}')

model_id = 'Salesforce/codet5p-220m'

print(f'{model_id} tokenizer yükleniyor...')
try:
    tokenizer = RobertaTokenizer.from_pretrained(model_id)
except Exception as e:
    print(f"Hata alındı, güvenli mod deneniyor: {e}")
    tokenizer = RobertaTokenizer.from_pretrained(model_id, extra_special_tokens=None)

print(f'✅ Tokenizer yüklendi! ({len(tokenizer)} token)')

print('CodeT5+ modeli yükleniyor...')
model = T5ForConditionalGeneration.from_pretrained(model_id).to(device)
print(f'✅ Model yüklendi!')


Cihaz: CUDA
Salesforce/codet5p-220m tokenizer yükleniyor...
Hata alındı, güvenli mod deneniyor: extra_special_tokens must be a list/tuple of str or AddedToken, or a dict mapping names to tokens
✅ Tokenizer yüklendi! (32100 token)
CodeT5+ modeli yükleniyor...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model yüklendi!


In [6]:
# ============================================================
# ADIM 5: Veriyi Temizle ve Tokenize Et
# ============================================================

# Önce sütun isimlerini belirle (buggy/fixed veya input/output olabilir)
ornek = dataset['train'][0]
BUGGY_SUTUN = 'buggy' if 'buggy' in ornek else 'input'
FIXED_SUTUN = 'fixed' if 'fixed' in ornek else 'output'
print(f'Sütunlar: {BUGGY_SUTUN} → {FIXED_SUTUN}')

def temizle_ve_tokenize(examples):
    # Giriş: hatalı kod
    girdi = [
        f'Fix this code: {str(b)[:400]}'
        for b in examples[BUGGY_SUTUN]
    ]
    # Hedef: düzeltilmiş kod
    hedef = [str(f)[:200] for f in examples[FIXED_SUTUN]]

    model_inputs = tokenizer(
        girdi, max_length=512,
        padding='max_length', truncation=True
    )
    labels = tokenizer(
        text_target=hedef, max_length=256,
        padding='max_length', truncation=True
    )

    # Padding token'larını -100 ile maskele (kritik!)
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Veri tokenize ediliyor...')
sutunlar = dataset['train'].column_names

tok_train = dataset['train'].map(
    temizle_ve_tokenize, batched=True,
    remove_columns=sutunlar
)
tok_eval = dataset['validation'].map(
    temizle_ve_tokenize, batched=True,
    remove_columns=dataset['validation'].column_names
)

print(f'✅ Tokenize tamamlandı!')
print(f'   Train: {len(tok_train)} örnek')
print(f'   Eval : {len(tok_eval)} örnek')

Sütunlar: buggy → fixed
Veri tokenize ediliyor...
✅ Tokenize tamamlandı!
   Train: 46680 örnek
   Eval : 5835 örnek


In [10]:
import torch
from transformers.optimization import Adafactor
from torch.utils.data import DataLoader

# Verileri formatla
tok_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Batch size 8 olarak kalıyor (Hız sağlar)
train_loader = DataLoader(tok_train, batch_size=8, shuffle=True)

# Adafactor (T5 için en güvenli optimizer)
optimizer = Adafactor(
    model.parameters(),
    lr=1e-5,
    eps=(1e-30, 1e-3),
    clip_threshold=1.0,
    decay_rate=-0.8,
    beta1=None,
    weight_decay=0.0,
    relative_step=False,
    scale_parameter=False,
    warmup_init=False
)

EPOCHS = 2
print(f'🚀 Kararlı Eğitim Başlıyor (Batch Size: 8, fp32)...')

for epoch in range(EPOCHS):
    model.train()
    toplam_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        # Autocast/Scaler kısmını kaldırdık (Hata kaynağı buydu)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        if torch.isnan(loss):
            continue

        loss.backward()
        optimizer.step()

        toplam_loss += loss.item()

        if step % 50 == 0:
            print(f'  Epoch {epoch+1} | Adım {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    print(f'✅ Epoch {epoch+1} tamamlandı.')

# Kaydet
model.save_pretrained(REPAIR_YOLU)
tokenizer.save_pretrained(REPAIR_YOLU)
print(f'\n✅ Model kaydedildi: {REPAIR_YOLU}')


🚀 Kararlı Eğitim Başlıyor (Batch Size: 8, fp32)...
  Epoch 1 | Adım 0/5835 | Loss: 1.1641
  Epoch 1 | Adım 50/5835 | Loss: 0.2637
  Epoch 1 | Adım 100/5835 | Loss: 0.2440
  Epoch 1 | Adım 150/5835 | Loss: 0.2240
  Epoch 1 | Adım 200/5835 | Loss: 0.1669
  Epoch 1 | Adım 250/5835 | Loss: 0.2402
  Epoch 1 | Adım 300/5835 | Loss: 0.1937
  Epoch 1 | Adım 350/5835 | Loss: 0.1801
  Epoch 1 | Adım 400/5835 | Loss: 0.1816
  Epoch 1 | Adım 450/5835 | Loss: 0.2944
  Epoch 1 | Adım 500/5835 | Loss: 0.2266
  Epoch 1 | Adım 550/5835 | Loss: 0.1708
  Epoch 1 | Adım 600/5835 | Loss: 0.3271
  Epoch 1 | Adım 650/5835 | Loss: 0.2244
  Epoch 1 | Adım 700/5835 | Loss: 0.1801
  Epoch 1 | Adım 750/5835 | Loss: 0.1547
  Epoch 1 | Adım 800/5835 | Loss: 0.2057
  Epoch 1 | Adım 850/5835 | Loss: 0.1921
  Epoch 1 | Adım 900/5835 | Loss: 0.1577
  Epoch 1 | Adım 950/5835 | Loss: 0.1636
  Epoch 1 | Adım 1000/5835 | Loss: 0.2286
  Epoch 1 | Adım 1050/5835 | Loss: 0.1432
  Epoch 1 | Adım 1100/5835 | Loss: 0.1699
  Epoc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model kaydedildi: /content/drive/MyDrive/CodeReviewBot/model_ciktisi/repair_model


In [11]:
# ============================================================
# ADIM 7: Test Et
# ============================================================
model.eval()

def kodu_duzeltt(buggy_kod):
    prompt = f'Fix this code: {buggy_kod[:400]}'
    inputs = tokenizer(
        prompt, return_tensors='pt',
        max_length=512, truncation=True
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256, min_length=5,
            num_beams=4, no_repeat_ngram_size=3,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test örnekleri
test_kodlari = [
    'def divide(a, b):\n    return a / b',
    'def factorial(n):\n    return n * factorial(n-1)',
]

print('🧪 REPAIR MODEL TEST SONUÇLARI')
print('='*60)
for kod in test_kodlari:
    duzeltme = kodu_duzeltt(kod)
    print(f'\nHatalı Kod : {kod}')
    print(f'Düzeltme   : {duzeltme}')
    print('-'*60)

🧪 REPAIR MODEL TEST SONUÇLARI

Hatalı Kod : def divide(a, b):
    return a / b
Düzeltme   : public static double divide(double a , double b ) { return a / b ; } 

------------------------------------------------------------

Hatalı Kod : def factorial(n):
    return n * factorial(n-1)
Düzeltme   : public static double factorial(double n ) { return n * n ; } 

------------------------------------------------------------


In [12]:
# ============================================================
# ADIM 8: Drive Yapısını Doğrula
# ============================================================
import os

print('Drive yapısı kontrol ediliyor...')
model_ciktisi = f'{DRIVE_KOKU}/model_ciktisi'
for item in os.listdir(model_ciktisi):
    print(f'  ├── {item}')

print('\n✅ Tamamlandı!')
print('  final_model/  → Code Review modeli (korundu)')
print('  repair_model/ → Code Repair modeli (yeni eklendi)')

Drive yapısı kontrol ediliyor...
  ├── final_model
  ├── faiss_index.bin
  ├── corpus_data.pkl
  ├── metrikler.json
  ├── repair_model

✅ Tamamlandı!
  final_model/  → Code Review modeli (korundu)
  repair_model/ → Code Repair modeli (yeni eklendi)
